# Day 3 — Memory, RAG and MCP

## Objectives

1. Understand conversation memory.
2. Compare window buffer, summary, and vector memory.
3. Build a small RAG pipeline.
4. Retrieve relevant chunks and answer with citations.
5. Build a small MCP server.
6. Expose two tools and one resource.
7. Connect an MCP client.
8. Discover and invoke MCP tools.

---

## Architecture

User
  ↓
Agent
  ├── Conversation Memory
  ├── RAG
  └── MCP Tools
        ↓
    External capabilities

## 1. Window Buffer Memory

Window buffer memory keeps the most recent conversation messages.

Example:

Message 1
Message 2
Message 3
Message 4
Message 5

If the window size is 3:

Message 3
Message 4
Message 5

Older messages are removed from the active context.

In [1]:
conversation = [
    {"role": "user", "content": "My name is Arun."},
    {"role": "assistant", "content": "Nice to meet you, Arun."},
    {"role": "user", "content": "I am learning agentic AI."},
    {"role": "assistant", "content": "Great! Agentic AI is about systems that can reason and act."},
    {"role": "user", "content": "I am currently learning RAG."},
    {"role": "assistant", "content": "RAG combines retrieval with generation."},
]

window_size = 3

recent_messages = conversation[-window_size:]

for message in recent_messages:
    print(f"{message['role']}: {message['content']}")

assistant: Great! Agentic AI is about systems that can reason and act.
user: I am currently learning RAG.
assistant: RAG combines retrieval with generation.


## 2. Summary Memory

Instead of keeping every previous message, we can maintain a compact summary.

Example:

Original conversation:
- User introduced themselves.
- User is learning agentic AI.
- User is learning RAG.
- User prefers Python.
- User completed an agent-core exercise.

Summary:

"The user is learning agentic AI and RAG using Python and has
completed an agent-core exercise."

The summary can be placed into the model context instead of the
entire conversation.

In [2]:
conversation_summary = """
The user is learning agentic AI and RAG using Python.
The user has completed an agent-core exercise.
"""

print(conversation_summary.strip())

The user is learning agentic AI and RAG using Python.
The user has completed an agent-core exercise.


## 3. Vector Memory

Vector memory stores memories as embeddings.

When a new request arrives:

User request
    ↓
Embedding
    ↓
Similarity search
    ↓
Relevant memories
    ↓
Add them to context

Example memories:

1. User prefers Python.
2. User is building an AI assistant.
3. User uses PostgreSQL.
4. User is learning MCP.

Query:

"What language should I use for this implementation?"

Semantic retrieval can identify:

"User prefers Python."

Unlike a window buffer, vector memory does not depend only on
recency. It retrieves memories based on semantic relevance.

In [3]:
memory_types = {
    "Window Buffer": "Keeps recent messages",
    "Summary": "Compresses conversation into a shorter representation",
    "Vector Memory": "Retrieves semantically relevant memories",
}

for name, description in memory_types.items():
    print(f"{name}: {description}")

Window Buffer: Keeps recent messages
Summary: Compresses conversation into a shorter representation
Vector Memory: Retrieves semantically relevant memories


# Part 2 — RAG Pipeline

RAG = Retrieval-Augmented Generation.

Pipeline:

Document
   ↓
Load
   ↓
Chunk
   ↓
Embed
   ↓
Store
   ↓
Retrieve
   ↓
Relevant chunks
   ↓
LLM
   ↓
Answer + Citations

## RAG Source Document

We will create a small artificial document so that the entire
pipeline is easy to understand and reproducible.

In [4]:
document = """
DocuChat is a RAG-powered application for asking questions about documents.

DocuChat stores document chunks and their embeddings in PostgreSQL
using the pgvector extension.

The retrieval process converts the user question into an embedding
and compares it with stored document embeddings.

The most relevant chunks are then provided to the language model as
context.

The final answer should be grounded in the retrieved document
content and should include citations to the source chunks.
"""

print(document)


DocuChat is a RAG-powered application for asking questions about documents.

DocuChat stores document chunks and their embeddings in PostgreSQL
using the pgvector extension.

The retrieval process converts the user question into an embedding
and compares it with stored document embeddings.

The most relevant chunks are then provided to the language model as
context.

The final answer should be grounded in the retrieved document
content and should include citations to the source chunks.



## Step 1 — Load

Loading means reading source documents into the application.

For this exercise our document is already available as a Python string.
In a real RAG application, the loader could read:

- PDF
- TXT
- Markdown
- DOCX
- HTML
- Database records

In [5]:
def chunk_text(text, chunk_size=250):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


chunks = chunk_text(document, chunk_size=40)

print(f"Total chunks: {len(chunks)}")

for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

Total chunks: 2

--- Chunk 1 ---
DocuChat is a RAG-powered application for asking questions about documents. DocuChat stores document chunks and their embeddings in PostgreSQL using the pgvector extension. The retrieval process converts the user question into an embedding and compares it with stored document embeddings.

--- Chunk 2 ---
The most relevant chunks are then provided to the language model as context. The final answer should be grounded in the retrieved document content and should include citations to the source chunks.


## Step 3 — Embed

An embedding converts text into a numerical vector.

Text
 ↓
Embedding model
 ↓
Vector

We will use a small sentence-transformers model for this exercise.

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks)

print("Number of chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 2
Embedding shape: (2, 384)


In [7]:
vector_store = []

for i, (chunk, embedding) in enumerate(
    zip(chunks, chunk_embeddings),
    start=1
):
    vector_store.append({
        "chunk_id": i,
        "text": chunk,
        "embedding": embedding,
        "source": "docuchat_demo.txt",
    })

print("Stored chunks:", len(vector_store))

Stored chunks: 2


## Step 5 — Retrieve

User question:

"Where are the embeddings stored?"

Process:

Question
   ↓
Question embedding
   ↓
Similarity comparison
   ↓
Top-k chunks

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

query = "Where are the embeddings stored?"

query_embedding = embedding_model.encode([query])

scores = cosine_similarity(
    query_embedding,
    [item["embedding"] for item in vector_store]
)[0]

top_k = 2

ranked_indices = scores.argsort()[::-1][:top_k]

retrieved_chunks = []

for index in ranked_indices:
    item = vector_store[index]

    retrieved_chunks.append({
        "chunk_id": item["chunk_id"],
        "text": item["text"],
        "source": item["source"],
        "score": float(scores[index]),
    })

for result in retrieved_chunks:
    print("=" * 60)
    print("Chunk ID:", result["chunk_id"])
    print("Score:", result["score"])
    print("Source:", result["source"])
    print(result["text"])

Chunk ID: 1
Score: 0.24916133284568787
Source: docuchat_demo.txt
DocuChat is a RAG-powered application for asking questions about documents. DocuChat stores document chunks and their embeddings in PostgreSQL using the pgvector extension. The retrieval process converts the user question into an embedding and compares it with stored document embeddings.
Chunk ID: 2
Score: 0.23148801922798157
Source: docuchat_demo.txt
The most relevant chunks are then provided to the language model as context. The final answer should be grounded in the retrieved document content and should include citations to the source chunks.


In [9]:
answer = (
    "The embeddings are stored in PostgreSQL using the pgvector extension."
)

citations = [
    {
        "source": result["source"],
        "chunk_id": result["chunk_id"],
        "score": result["score"],
    }
    for result in retrieved_chunks
]

print("Answer:")
print(answer)

print("\nCitations:")
for citation in citations:
    print(
        f"[{citation['chunk_id']}] "
        f"{citation['source']} "
        f"(score={citation['score']:.3f})"
    )

Answer:
The embeddings are stored in PostgreSQL using the pgvector extension.

Citations:
[1] docuchat_demo.txt (score=0.249)
[2] docuchat_demo.txt (score=0.231)


# MCP Architecture

MCP separates the AI application from external capabilities.

Host / Application
        ↓
MCP Client
        ↓
MCP Server
   ┌────┴────┐
   ↓         ↓
 Tools    Resources

In this exercise:

Tools:
1. add
2. multiply

Resource:
info://project

The client discovers the available capabilities and invokes the
appropriate tool.